# 05b · Model eğitimi — kalan foldlar 1-4 (Kaggle, GPU)

05 ile aynı; yalnızca fold 1-4 çalışır. 05 (fold 0) ile birlikte 5 foldluk ensemble ve 58 altın vakanın tamamı için OOF verir.

**Bu notebook ne yapar?** Cache'teki hacimler ve `02_etiketler` çıktısı (`train_labels.csv`)
ile vaka düzeyi modeli eğitir (etiket v2: `train_labels_v2.csv`); her fold için checkpoint, OOF tahmini ve eğitim geçmişi yazar.

**Model (`src/model.py`):**
```
her slottan 2.5D pencereler [N,3,224,224]
   → ConvNeXt-Tiny (ImageNet-22k ön eğitimli)       → öznitelik [N,768]
   + slot gömmesi (düzlem × sekans) + konum gömmesi (yığının neresi)
   → 1 katman Transformer (pencereler arası bağlam)
   → 12 başlı bulguya özgü dikkat havuzlama + ortalama havuzlama artığı
   → 12 logit
```

**Eğitim (`src/engine.py`):** yumuşak etiketlerle BCE, altın vakalara 2× ağırlık, AdamW,
ısınmalı kosinüs öğrenme oranı, fp16 AMP, gradyan kırpma. Veri artırma: slot bazında küçük
dönme/ölçek/kaydırma + parlaklık/kontrast. **Yatay çevirme yok** (medial/lateral bozulur).

**Doğrulama:** her epoch `pseudo_auc` (doğrulama fold'undaki rapor etiketlerine göre) ve
`gold_auc` (fold'daki ~12 altın vaka) raporlanır. En iyi epoch `pseudo_auc`'a göre seçilir;
`gold_auc` tek fold'da çok gürültülüdür, 5 fold bitince 58 vakanın OOF'u ile yorumlanır.

**Kaggle ayarları:** Accelerator = GPU (T4 / P100), Internet = **açık** (ön eğitimli ağırlık
indirmek için), Input = `rsna-knee-src` + 04'ün çıktısı (cache).

In [ ]:
# Proje kökünü (src/ klasörünü içeren yer) Python yoluna ekle: yerelde ve Kaggle'da çalışır.
import os, sys
from pathlib import Path

def add_project_root():
    cands = [Path.cwd(), Path.cwd().parent]
    if Path("/kaggle/input").exists():
        # os.walk + budama: yarışmanın DICOM klasörlerine girmez (rglob dakikalar sürüyordu)
        for d, dirs, files in os.walk("/kaggle/input"):
            dirs[:] = [x for x in dirs if x not in ("train_series", "test_series")]
            if Path(d).name == "src" and "__init__.py" in files:
                cands.append(Path(d).parent)
    for c in cands:
        if (c / "src" / "__init__.py").exists():
            sys.path.insert(0, str(c))
            return c
    raise FileNotFoundError("src/ bulunamadı")

PROJECT_ROOT = add_project_root()
print("Proje kökü:", PROJECT_ROOT)

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.config import EXPERIMENTS_DIR, IS_KAGGLE, TARGET_COLS
from src.dataset import KneeCacheDataset, collate_studies
from src.engine import cosine_with_warmup, evaluate, make_folds, predict, train_one_epoch
from src.model import build_model

CFG = {
    "backbone": "convnext_tiny.fb_in22k_ft_in1k",
    "context_layers": 1,
    "drop_path": 0.1,
    "epochs": 8,
    "batch_size": 2,          # vaka / adım (~58 pencere; T4 16 GB belleğe sığması için)
    "grad_accum": 4,          # etkin batch = 8 vaka
    "lr_backbone": 1e-4,
    "lr_head": 5e-4,
    "weight_decay": 0.05,
    "warmup_frac": 0.1,
    "windows_per_slot": 6,    # eğitimde slot başına rastgele pencere
    "eval_windows_per_slot": 14,
    "gold_weight": 2.0,
    "n_folds": 5,
    "folds_to_run": [1, 2, 3, 4],  # 05b: kalan foldlar (fold 0 = 05)
    "seed": 42,
    "num_workers": 4,
}
DRY_RUN = not IS_KAGGLE
if DRY_RUN:  # yerel deneme: küçük omurga, 1 epoch, 2 fold
    CFG.update(backbone="resnet18", n_folds=2, epochs=1, batch_size=2, grad_accum=1, num_workers=0,
               windows_per_slot=2, eval_windows_per_slot=4)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(CFG["seed"]); np.random.seed(CFG["seed"])
print("Cihaz:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "")

## 1. Girdileri bul: cache klasörü ve etiket tablosu

In [ ]:
from src.config import find_input

def find_one(name: str, roots):
    for r in roots:
        if Path(r).exists():
            hit = find_input(name, str(r))   # DICOM klasörlerine girmeyen arama
            if hit is not None:
                return hit
    raise FileNotFoundError(name)

roots = ["/kaggle/input"] if IS_KAGGLE else [EXPERIMENTS_DIR / "cache", PROJECT_ROOT / "data"]
CACHE_DIR = find_one("cache_config.json", roots).parent
LABELS_PATH = find_one("train_labels_v2.csv", roots)  # etiket v2 (altın AUC 0.9014)
cache_cfg = json.loads((CACHE_DIR / "cache_config.json").read_text())
print("Cache   :", CACHE_DIR, cache_cfg)
print("Etiket  :", LABELS_PATH)

labels = pd.read_csv(LABELS_PATH)
if DRY_RUN:
    # Yerelde cache'te yalnızca 1 test vakası var. Kodun uçtan uca çalıştığını görmek için
    # onu 12 eğitim vakasının (6'sı altın) adıyla kopyalıyoruz. Metrikler anlamsızdır.
    import shutil
    src_npz = sorted(CACHE_DIR.glob("*.npz"))[0]
    fake_dir = EXPERIMENTS_DIR / "dryrun_cache"
    fake_dir.mkdir(exist_ok=True)
    labels = pd.concat([labels[labels.is_gold].head(6), labels[~labels.is_gold].head(6)])
    for s in labels.StudyInstanceUID:
        shutil.copy(src_npz, fake_dir / f"{s}.npz")
    CACHE_DIR = fake_dir
cached = {p.stem for p in CACHE_DIR.glob("*.npz")}
missing = (~labels.StudyInstanceUID.isin(cached)).sum()
print(f"Etiketli vaka: {len(labels)} | cache'te olmayan: {missing}")
labels = labels[labels.StudyInstanceUID.isin(cached)].reset_index(drop=True)
folds = make_folds(labels, CFG["n_folds"], CFG["seed"])
folds.groupby("fold").agg(vaka=("StudyInstanceUID", "size"), altin=("is_gold", "sum"))

## 2. Eğitim döngüsü

Omurga ve başlık için ayrı öğrenme oranları: ön eğitimli omurga küçük adımlarla (1e-4),
sıfırdan başlayan dikkat/sınıflandırıcı katmanları daha büyük adımlarla (5e-4) öğrenir.

In [ ]:
OUT = EXPERIMENTS_DIR / "model_out"
OUT.mkdir(parents=True, exist_ok=True)

def run_fold(fold: int) -> dict:
    tr = folds[folds.fold != fold].reset_index(drop=True)
    va = folds[folds.fold == fold].reset_index(drop=True)
    dl_tr = DataLoader(KneeCacheDataset(tr, CACHE_DIR, train=True, windows_per_slot=CFG["windows_per_slot"]),
                       batch_size=CFG["batch_size"], shuffle=True, num_workers=CFG["num_workers"],
                       collate_fn=collate_studies, pin_memory=True, drop_last=True,
                       persistent_workers=CFG["num_workers"] > 0)
    dl_va = DataLoader(KneeCacheDataset(va, CACHE_DIR, train=False, eval_windows_per_slot=CFG["eval_windows_per_slot"]),
                       batch_size=CFG["batch_size"], shuffle=False, num_workers=CFG["num_workers"],
                       collate_fn=collate_studies, pin_memory=True)

    model = build_model(CFG, pretrained=True).to(device)
    bb = [p for n, p in model.named_parameters() if n.startswith("backbone.")]
    hd = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]
    opt = torch.optim.AdamW([{"params": bb, "lr": CFG["lr_backbone"]},
                             {"params": hd, "lr": CFG["lr_head"]}], weight_decay=CFG["weight_decay"])
    steps = CFG["epochs"] * max(1, len(dl_tr) // CFG["grad_accum"])
    sch = cosine_with_warmup(opt, max(1, int(CFG["warmup_frac"] * steps)), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

    best, history = -1.0, []
    for ep in range(1, CFG["epochs"] + 1):
        t = time.time()
        loss = train_one_epoch(model, dl_tr, opt, sch, scaler, device, CFG["gold_weight"],
                               CFG["grad_accum"], log_every=100)
        pred = predict(model, dl_va, device)
        m = evaluate(pred, folds)
        history.append({"epoch": ep, "loss": loss, **m, "sec": time.time() - t})
        flag = ""
        score = m["pseudo_auc"] if not np.isnan(m["pseudo_auc"]) else 0.0
        if score > best:
            best = score
            flag = "  <- en iyi, kaydedildi"
            torch.save({"model": model.state_dict(), "config": CFG, "cache_config": cache_cfg,
                        "fold": fold, "epoch": ep, "metrics": m}, OUT / f"model_fold{fold}.pt")
            pred.to_csv(OUT / f"oof_fold{fold}.csv", index=False)
        print(f"fold {fold} ep {ep}: loss {loss:.4f} | pseudo_auc {m['pseudo_auc']:.4f} | "
              f"gold_auc {m['gold_auc']:.4f} (n={m['n_gold']}) | {time.time() - t:.0f}s{flag}", flush=True)
    json.dump(history, open(OUT / f"history_fold{fold}.json", "w"), indent=2)
    del model, opt
    torch.cuda.empty_cache()
    return {"fold": fold, "best_pseudo_auc": best}

results = [run_fold(f) for f in CFG["folds_to_run"]]
pd.DataFrame(results)

## 3. OOF değerlendirmesi

Çalışan tüm fold'ların OOF tahminleri birleştirilip bulgu bazında AUC hesaplanır.
Altın vaka AUC'si yalnızca 5 fold tamamlanınca (58 vakanın hepsi OOF olunca) anlamlıdır.

In [ ]:
from src.engine import macro_auc

oof = pd.concat([pd.read_csv(p) for p in sorted(OUT.glob("oof_fold*.csv"))])
m = folds.set_index("StudyInstanceUID").loc[oof.StudyInstanceUID]
per_class = pd.DataFrame({
    "pseudo": macro_auc(m[TARGET_COLS].values, oof[TARGET_COLS].values),
})
g = m.is_gold.values.astype(bool)
if g.sum() >= 5:
    per_class["gold"] = pd.Series(macro_auc(m[TARGET_COLS].values[g], oof[TARGET_COLS].values[g]))
print(f"OOF vaka: {len(oof)} | altın: {g.sum()}")
per_class.T.round(3)

**Sonraki adım:** Bu notebook'un çıktısını (`model_out/*.pt`) `06_submission_kaggle.ipynb`'e Input olarak ekleyin.